Перед запуском скрипта необходимо убедиться, что создана новая копия файла result.xlsx и указать в параметрах запуска референсную и изменяемую колонки, референсное значение и суффикс. Скрипт записывает в копию файла result изменённую колонку для копирования.

In [ ]:
import pandas as pd
from openpyxl.styles import Alignment

def update_excel_cells(file_path, target_col, filter_col, filter_value, suffix, output_path):
    # 1. Загрузка и обработка данных
    df = pd.read_excel(file_path)
    
    def process_row(row):
        current_val = str(row[target_col]).strip() if pd.notna(row[target_col]) else ""
        condition_val = str(row[filter_col])
        
        if str(filter_value) in str(condition_val) and suffix not in current_val:
            if current_val:
                return f"{current_val}; {suffix}"
            else:
                return suffix
        return row[target_col]

    df[target_col] = df.apply(process_row, axis=1)
    
    # 2. Сохранение только нужной колонки через движок openpyxl
    writer = pd.ExcelWriter(output_path, engine='openpyxl')
    df[[target_col]].to_excel(writer, index=False, sheet_name='Sheet1')
    
    # 3. Применение форматирования
    workbook = writer.book
    worksheet = writer.sheets['Sheet1']
    
    # Устанавливаем ширину колонки (100 — это очень широко, почти на весь экран)
    worksheet.column_dimensions['A'].width = 100
    
    # Настраиваем стиль для всех ячеек в колонке A (кроме заголовка, если нужно)
    alignment_style = Alignment(
        wrap_text=True,      # Перенос текста
        vertical='center',      # Вертикальное выравнивание по верхнему краю
        horizontal='left'    # Горизонтальное по левому (опционально)
    )
    
    for row in range(1, worksheet.max_row + 1):
        cell = worksheet.cell(row=row, column=1)
        cell.alignment = alignment_style
        # Автоподбор высоты в Excel обычно срабатывает сам при wrap_text=True, 
        # но мы не ограничиваем высоту строки, чтобы она расширялась под текст.
        worksheet.row_dimensions[row].height = None 

    writer.close()
    print(f"Файл '{output_path}' готов и отформатирован.")

# --- НАСТРОЙКИ ---
settings = {
    "file_path": "result — копия.xlsx",        # Путь к исходному файлу
    "target_col": "ДОПОЛНИТЕЛЬНЫЕ ПАРАМЕТРЫ",        # Колонку, КУДА дописываем
    "filter_col": "Прямоугольная зона приготовления",          # Колонка, ГДЕ проверяем условие
    "filter_value": "Да",       # Какое значение ищем в колонке-фильтре
    "suffix": "Прямоугольная зона приготовления",         # Что именно дописываем
    "output_path": "result — копия.xlsx" # Имя итогового файла
}

update_excel_cells(**settings)


Файл 'result — копия.xlsx' готов и отформатирован.
